 imports and load data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ARTIFACTS = Path("artifacts")
PLOTS = ARTIFACTS / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

clean = pd.read_parquet(ARTIFACTS / "cleaned")
anomalies = pd.read_parquet(ARTIFACTS / "anomalies_zscore")

clean["ts"] = pd.to_datetime(clean["ts"])
anomalies["ts"] = pd.to_datetime(anomalies["ts"])
print("Clean rows:", len(clean))
print("Z-score anomalies:", len(anomalies))


Cell 2 — anomalies per region and severity

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=anomalies, x="region", hue="severity", order=sorted(anomalies.region.unique()))
plt.title("Z-score anomalies by region and severity")
plt.xlabel("Region")
plt.ylabel("Anomaly count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(PLOTS / "anomalies_per_region.png", dpi=160)
plt.show()


Cell 3 — one flagged point with its expected baseline

In [ ]:
if anomalies.empty:
    print("No anomalies to plot.")
else:
    example = anomalies.sort_values("z_score", key=lambda s: s.abs(), ascending=False).iloc[0]
    region = example["region"]
    ts = example["ts"]
    series = clean[
        (clean["region"] == region)
        & (clean["ts"] >= ts - pd.Timedelta(days=3))
        & (clean["ts"] <= ts + pd.Timedelta(days=3))
    ].sort_values("ts")

    plt.figure(figsize=(12, 5))
    plt.plot(series["ts"], series["mw"], label="Observed MW")
    plt.axvline(ts, color="red", linestyle="--", label="Flagged timestamp")
    plt.axhline(example["expected_mw"], color="orange", linestyle=":", label="Context expected MW")
    plt.scatter([ts], [example["mw"]], color="red", zorder=3)
    plt.title(f"{region}: observed MW vs. contextual expected MW")
    plt.xlabel("Timestamp")
    plt.ylabel("MW")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS / "example_anomaly.png", dpi=160)
    plt.show()


Cell 4 — anomaly counts by hour

In [ ]:
anomalies["hour"] = anomalies["ts"].dt.hour
by_hour = anomalies.groupby("hour").size().reindex(range(24), fill_value=0)

plt.figure(figsize=(10, 4))
sns.barplot(x=by_hour.index, y=by_hour.values, color="steelblue")
plt.title("Z-score anomalies by hour of day")
plt.xlabel("Hour")
plt.ylabel("Anomaly count")
plt.tight_layout()
plt.savefig(PLOTS / "anomalies_by_hour.png", dpi=160)
plt.show()


Cell 5 — event-date alignment check

events = {
    "2003 Northeast blackout": ("2003-08-14", "2003-08-15"),
    "June 2012 derecho": ("2012-06-28", "2012-07-01"),
    "Hurricane Sandy": ("2012-10-29", "2012-10-31"),
    "January 2014 polar vortex": ("2014-01-05", "2014-01-08"),
}

for name, (start, end) in events.items():
    mask = anomalies["ts"].between(pd.Timestamp(start), pd.Timestamp(end))
    hits = anomalies.loc[mask].sort_values("ts")
    print(f"\n{name}: {len(hits)} flagged rows across {hits['region'].nunique()} series")
    if not hits.empty:
        print(hits[["region", "ts", "mw", "expected_mw", "z_score", "severity"]].head(20).to_string(index=False))
